In [ ]:
%cd ..

In [ ]:
from pathlib import Path
import pandas as pd
from molrgen.data.pydantic_dataset import read_jsonl
from tqdm.auto import tqdm

queries = read_jsonl(Path("data/molgendata/test_data/test_prompts_ood_boxed.jsonl"))
df_zinc = pd.read_csv("data/properties.csv")

In [ ]:
df_zinc.head()

In [ ]:
all_results = {
    i: [] for i in range(len(queries) // 100)
}

for i, query in enumerate(tqdm(queries)):
    smiles = df_zinc["smiles"].sample(128)
    for smi in smiles:
        all_results[i//100].append(
            dict(
                input=None,
                output="<answer> {} </answer>".format(smi),
                metadata = query.conversations[0].meta
            )
        )


In [ ]:
[len(r) for r in all_results.values()]

In [ ]:
# save to jsonl file
import json

output_dir = Path("MolGenOutput/test_ood/ZINC")
output_dir.mkdir(parents=True, exist_ok=True)

for idx in tqdm(all_results.keys()):
    output_path = output_dir / "ZINC_{}.jsonl".format(idx)
    with open(output_path, "w") as f:
        for result in all_results[idx]:
            f.write(json.dumps(result) + "\n")

In [ ]:
!wc -l MolGenOutput/test_ood/ZINC/*